# Multi-objective Acquisition: Pareto and Hypervolume Workflow

2目的を同時最大化するMOBOを、`ModelListGP`からPareto front、reference point、qLogEHVI、joint batch candidate、評価、Pareto更新まで接続します。

関連理論:

- [Multi-objective acquisition](../../docs/theory/acquisition/07_multiobjective.md)
- [Multi-task / multi-output](../../docs/theory/07_multitask_multioutput.md)

```text
multi-output posterior
        ↓
Pareto structure + reference point
        ↓
qLogEHVI / qLogNEHVI / qLogNParEGO
        ↓
joint candidate optimization
        ↓
objective evaluation
        ↓
updated Pareto front
```

## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.multi_objective.logei import (
    qLogExpectedHypervolumeImprovement,
    qLogNoisyExpectedHypervolumeImprovement,
)
from botorch.acquisition.multi_objective.parego import qLogNParEGO
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    FastNondominatedPartitioning,
)
from botorch.utils.multi_objective.pareto import is_non_dominated

from robotorchan.models import ModelListGP, SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Two conflicting objectives

両目的をmaximize conventionへ揃えます。1つ目は`x≈0.25`、2つ目は`x≈0.75`付近を好むため、単一のbest pointではなくtrade-offを持つPareto setが生じます。

In [ ]:
def objectives(X: torch.Tensor) -> torch.Tensor:
    first = 1.0 - (X - 0.25).square()
    second = 1.0 - (X - 0.75).square()
    return torch.cat([first, second], dim=-1)


train_X = torch.linspace(0.0, 1.0, 9).unsqueeze(-1)
train_Y = objectives(train_X)
bounds = torch.tensor([[0.0], [1.0]])
print(train_X.shape, train_Y.shape)

## 3. Multi-output surrogate is not yet MOBO

`ModelListGP`は2出力posteriorを提供しますが、それだけではPareto optimizationにはなりません。どのoutputsをobjectivesとして扱うか、reference pointをどこに置くか、どのMO utilityを使うかはdecision layerの責務です。

In [ ]:
model = ModelListGP(
    SingleTaskGP(train_X, train_Y[:, [0]]),
    SingleTaskGP(train_X, train_Y[:, [1]]),
)
fit_gpytorch_mll(model.make_mll())
model.eval()

probe_X = torch.tensor([[0.3], [0.7]])
with torch.no_grad():
    posterior = model.posterior(probe_X)
print("posterior mean shape:", posterior.mean.shape)

## 4. Observed Pareto front

In [ ]:
pareto_mask = is_non_dominated(train_Y)
pareto_Y = train_Y[pareto_mask]
pareto_X = train_X[pareto_mask]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(train_Y[:, 0], train_Y[:, 1], label="observations")
ax.scatter(pareto_Y[:, 0], pareto_Y[:, 1], marker="x", s=80, label="Pareto")
ax.set_xlabel("objective 1")
ax.set_ylabel("objective 2")
ax.legend()
plt.show()
print("Pareto X:", pareto_X.squeeze(-1))

## 5. Reference point is part of the problem definition

このsynthetic exampleでは観測objectiveの各成分最小値より0.1悪い点を使います。これは一般則ではありません。実問題ではobjective transform後の尺度や「hypervolumeとして数えたい最低水準」から設計します。

In [ ]:
ref_point = train_Y.min(dim=0).values - 0.1
partitioning = FastNondominatedPartitioning(ref_point=ref_point, Y=train_Y)
print("reference point:", ref_point)

## 6. qLogEHVI and joint q=2 candidate generation

q=2はpointwise EHVI上位2点ではありません。2候補を同時に追加したときの**joint hypervolume improvement**をposterior samplesで評価します。

In [ ]:
ehvi = qLogExpectedHypervolumeImprovement(
    model=model,
    ref_point=ref_point.tolist(),
    partitioning=partitioning,
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=123),
)

candidates, ehvi_value = optimize_acqf(
    acq_function=ehvi,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=64,
)

assert candidates.shape == torch.Size([2, 1])
print("joint candidates:", candidates.squeeze(-1))
print("qLogEHVI value:", ehvi_value)

## 7. Evaluate the batch and update the Pareto front

In [ ]:
candidate_Y = objectives(candidates)
updated_X = torch.cat([train_X, candidates], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)
updated_mask = is_non_dominated(updated_Y)
updated_pareto_Y = updated_Y[updated_mask]

print("candidate objectives:\n", candidate_Y)
print("Pareto points before / after:", pareto_Y.shape[0], updated_pareto_Y.shape[0])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(train_Y[:, 0], train_Y[:, 1], label="initial observations")
ax.scatter(candidate_Y[:, 0], candidate_Y[:, 1], marker="x", s=90, label="new batch")
ax.scatter(
    updated_pareto_Y[:, 0],
    updated_pareto_Y[:, 1],
    facecolors="none",
    edgecolors="black",
    s=100,
    label="updated Pareto",
)
ax.set_xlabel("objective 1")
ax.set_ylabel("objective 2")
ax.legend()
plt.show()

## 8. qLogNEHVI: uncertainty in the baseline front

noiseがあるとobserved Pareto frontとlatent Pareto frontは一致しません。qLogNEHVIは`X_baseline`のlatent objective uncertaintyも積分します。ここではAPIとcandidate pathを同じmodelで実行確認します。

In [ ]:
nehvi = qLogNoisyExpectedHypervolumeImprovement(
    model=model,
    ref_point=ref_point.tolist(),
    X_baseline=train_X,
    sampler=SobolQMCNormalSampler(torch.Size([32]), seed=456),
    prune_baseline=False,
)
nehvi_candidate, nehvi_value = optimize_acqf(
    acq_function=nehvi,
    bounds=bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
)
print("qLogNEHVI candidate:", nehvi_candidate)
print("qLogNEHVI value:", nehvi_value)

## 9. qLogNParEGO: scalarization-based Pareto exploration

NParEGOはhypervolumeを直接最適化せず、multi-output posteriorをscalarized subproblemへ写します。iterationごとにweightsを変えることでfrontの異なる領域を探索します。

In [ ]:
weights = torch.tensor([0.35, 0.65])
nparego = qLogNParEGO(
    model=model,
    X_baseline=train_X,
    scalarization_weights=weights,
    sampler=SobolQMCNormalSampler(torch.Size([32]), seed=789),
)
nparego_candidate, nparego_value = optimize_acqf(
    acq_function=nparego,
    bounds=bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
)
print("weights:", weights)
print("qLogNParEGO candidate:", nparego_candidate)
print("qLogNParEGO value:", nparego_value)

## 10. The three decision semantics

```text
qLogEHVI
    current Pareto frontへのjoint hypervolume improvement

qLogNEHVI
    baseline objective / latent Pareto frontのuncertaintyも考慮

qLogNParEGO
    scalarization weightsで1つのsubproblemを作る
```

これらのacquisition valueを共通のperformance scoreとして比較しません。utility definitionが異なるためです。

## 11. Model dependence and candidate dependence are separate

このNotebookの`ModelListGP`は目的ごとに独立したGPです。一方、qLogEHVIのq=2 candidateはjoint posterior samplesで同時に価値付けされます。

- objective間dependence: surrogateが複数outputs間の相関をどうmodel化するか
- candidate間dependence: q候補のjoint posteriorをacquisitionがどう扱うか

この2つを混同しません。相関したobjectivesを明示的にmodel化したい場合はMultiTask/Kronecker系も候補です。

## 12. Important boundaries

- reference pointはtraining Yの最小値から常に自動決定できるものではありません。
- minimization objectiveは符号反転等でmaximize conventionへ揃えます。
- multi-output predictionとmulti-objective decisionは別概念です。
- black-box outcome constraintとknown candidate/input-space constraintは別レイヤーです。
- `X_pending`を含むasynchronous MOBOはBatch/Async Phaseで扱います。
- HVKGはTheory上存在しますが、現在robotorchanでintegration-testedなMO pathとは扱いません。
- many-objectiveではhypervolume partitioning costとreference-point sensitivityが増えます。

## 13. Related notebooks

- `05_model_list_gp.ipynb`: independent multi-output surrogate
- `04_multitask_gp.ipynb`: MultiTask / Kronecker posterior semantics
- `26_standard_acquisition.ipynb`: single-objective acquisition comparison

このNotebookはmulti-objective **decision workflow**を担当し、model-family説明を重複させません。